## Baseline

In [ ]:
import pandas as pd

In [ ]:
def read_file_as_df(file_name):
    import pandas as pd
    import csv

    import sys
    import pandas as pd

    maxInt = sys.maxsize

    while True:
        # decrease the maxInt value by factor 10
        # as long as the OverflowError occurs.

        try:
            csv.field_size_limit(maxInt)
            break
        except OverflowError:
            maxInt = int(maxInt/10)

    file = []
    col = []

    with open(file_name, encoding='latin-1') as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=';')
        line_count = 0
        for row in csv_reader:
            if line_count==0:
                for r in row:
                    col.append(r)
                line_count+=1
            else:
                line = []
                for r in row:
                    line.append(r)
                file.append(line)
                line_count += 1


    df = pd.DataFrame(file, columns = col)
    return df

In [ ]:
df = read_file_as_df('../data/e-SIC1BR.csv')

In [ ]:
df.head(1)

,age-bracket,gender,education,profession,req-text,resp-text,resp-type,clarity,service,1funct-request,...,55work-response,56achieve-response,57leisure-response,58home-response,59money-response,60relig-response,61death-response,62assent-response,63nonfl-response,64filler-response
0,a31-42,m,e2,academy,"Prezados, Gostaria de solicitar informações so...","Não dispomos de vaga em aberto, para o cargo d...",Acesso Concedido,5,5,"0,4531",...,"0,0612","0,102","0,0204",0,0,0,0,0,0,0


In [ ]:
df['age-bracket'].value_counts()

a31-42    16978
a17-30    16769
a43-xx    14015
Name: age-bracket, dtype: int64

In [ ]:
df[df['age-bracket'].isna()]

,age-bracket,gender,education,profession,req-text,resp-text,resp-type,clarity,service,1funct-request,...,55work-response,56achieve-response,57leisure-response,58home-response,59money-response,60relig-response,61death-response,62assent-response,63nonfl-response,64filler-response


In [ ]:
df[df['age-bracket']==""]

,age-bracket,gender,education,profession,req-text,resp-text,resp-type,clarity,service,1funct-request,...,55work-response,56achieve-response,57leisure-response,58home-response,59money-response,60relig-response,61death-response,62assent-response,63nonfl-response,64filler-response


In [ ]:
X = df['req-text']
y = pd.factorize(df['age-bracket'])[0]

In [ ]:
X.shape, len(X.unique())

((47762,), 41454)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=115, stratify=y)

In [ ]:
type(X_train)

pandas.core.series.Series

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index) + 1  # Add 1 for padding

2024-11-02 16:06:33.208412: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-11-02 16:06:33.208430: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


AttributeError: 'float' object has no attribute 'lower'

In [ ]:
vocab_size

66644

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

sequences = tokenizer.texts_to_sequences(X_train)
X_train = pad_sequences(sequences, maxlen=200)

test_sequences = tokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(test_sequences, maxlen=200)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=115, stratify=y_train)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.callbacks import EarlyStopping

input_length = 200  # sequence length
embedding_size = 128  # embedding dimension

lstm_units = 4  # number of LSTM cells

dropout_rate = 0.5  # dropout rate

epochs = 20  # number of epochs
batch_size = 128  # batch size

learning_rate = 0.01  # learning rate

validation_split = 0.25  # validation set ratio

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_size, input_length=input_length))
model.add(LSTM(lstm_units, return_sequences=False))
model.add(Dropout(dropout_rate))
model.add(Dense(3, activation='softmax'))

optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate) # qual optimizer usar?
model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 128)          8480256   
                                                                 
 lstm (LSTM)                 (None, 4)                 2128      
                                                                 
 dropout (Dropout)           (None, 4)                 0         


2024-10-23 23:44:59.643808: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-10-23 23:44:59.644398: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-10-23 23:44:59.644446: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2024-10-23 23:44:59.644491: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2024-10-23 23:44:59.644534: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Co

                                                                 
 dense (Dense)               (None, 3)                 15        
                                                                 
Total params: 8,482,399
Trainable params: 8,482,399
Non-trainable params: 0
_________________________________________________________________


In [ ]:
import numpy as np

y_train = np.array(y_train)
y_test = np.array(y_test)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

Epoch 1/20
224/224 [==============================] - 21s 87ms/step - loss: 1.0186 - accuracy: 0.4670 - val_loss: 0.9448 - val_accuracy: 0.5346
Epoch 2/20
224/224 [==============================] - 18s 82ms/step - loss: 0.8580 - accuracy: 0.5990 - val_loss: 0.9170 - val_accuracy: 0.5584
Epoch 3/20
224/224 [==============================] - 18s 82ms/step - loss: 0.7144 - accuracy: 0.6857 - val_loss: 0.9508 - val_accuracy: 0.5638
Epoch 4/20
224/224 [==============================] - 18s 82ms/step - loss: 0.6172 - accuracy: 0.7323 - val_loss: 0.9949 - val_accuracy: 0.5552
Epoch 5/20
224/224 [==============================] - 18s 82ms/step - loss: 0.5500 - accuracy: 0.7671 - val_loss: 1.0957 - val_accuracy: 0.5670
Epoch 6/20
224/224 [==============================] - 18s 82ms/step - loss: 0.4996 - accuracy: 0.7899 - val_loss: 1.1537 - val_accuracy: 0.5651
Epoch 7/20
224/224 [==============================] - 18s 81ms/step - loss: 0.4675 - accuracy: 0.8044 - val_loss: 1.2700 - val_accuracy:

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Validation Accuracy: {accuracy}')

299/299 [==============================] - 2s 7ms/step - loss: 1.2758 - accuracy: 0.5669
Validation Accuracy: 0.5669423341751099


In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

from sklearn.metrics import f1_score
f1_score(y_test, y_pred_classes, average='weighted')

299/299 [==============================] - 2s 7ms/step


0.5672392776287271

In [ ]:
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import GridSearchCV
import tensorflow as tf

def create_model(input_length = 200, embedding_size=128, lstm_units=4, dropout_rate=0.5, learning_rate=0.01):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_size, input_length=input_length))
    model.add(tf.keras.layers.LSTM(lstm_units, return_sequences=False))
    model.add(tf.keras.layers.Dropout(dropout_rate))
    model.add(tf.keras.layers.Dense(3, activation='softmax'))

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

In [ ]:
model_ = KerasClassifier(build_fn=create_model, epochs=20, batch_size=128)

param_grid = {
    'embedding_size': [128, 256],
    'lstm_units': [4, 16],
    'dropout_rate': [0.3, 0.5],
    'learning_rate': [0.001, 0.01],
}

grid = GridSearchCV(estimator=model_, param_grid=param_grid, cv=5, verbose=0)

grid_result = grid.fit(X_train, y_train, validation_data=(X_val, y_val), callbacks=[early_stopping])


In [ ]:
grid_result.best_params_, grid_result.best_score_

({'dropout_rate': 0.5,
  'embedding_size': 256,
  'learning_rate': 0.001,
  'lstm_units': 16},
 0.5749231100082397)

In [ ]:
f1 = []
for i in range(10):

    gs_model = create_model(input_length = 200, embedding_size=256, lstm_units=16, dropout_rate=0.5, learning_rate=0.001)

    history = gs_model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=128,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping]
    )

    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)

    f1.append(f1_score(y_test, y_pred_classes, average='weighted'))

print(f1)

Epoch 1/20
224/224 [==============================] - 81s 348ms/step - loss: 0.9965 - accuracy: 0.4863 - val_loss: 0.8879 - val_accuracy: 0.5688
Epoch 2/20
224/224 [==============================] - 79s 353ms/step - loss: 0.7447 - accuracy: 0.6831 - val_loss: 0.8692 - val_accuracy: 0.5879
Epoch 3/20
224/224 [==============================] - 78s 350ms/step - loss: 0.5110 - accuracy: 0.8100 - val_loss: 0.9718 - val_accuracy: 0.5880
Epoch 4/20
224/224 [==============================] - 77s 343ms/step - loss: 0.3419 - accuracy: 0.8829 - val_loss: 1.1629 - val_accuracy: 0.5865
Epoch 5/20
224/224 [==============================] - 71s 317ms/step - loss: 0.2436 - accuracy: 0.9213 - val_loss: 1.3496 - val_accuracy: 0.5764
Epoch 6/20
224/224 [==============================] - 35s 158ms/step - loss: 0.1827 - accuracy: 0.9425 - val_loss: 1.5187 - val_accuracy: 0.5801
Epoch 7/20
224/224 [==============================] - 35s 158ms/step - loss: 0.1512 - accuracy: 0.9529 - val_loss: 1.6925 - val_ac

In [ ]:
seeds = [11, 114, 115, 289, 42, 685, 383, 67]
f1=[]

X = df['req-text']
y = pd.factorize(df['age-bracket'])[0]

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(X_train)

    vocab_size = len(tokenizer.word_index) + 1

    sequences = tokenizer.texts_to_sequences(X_train)
    X_train = pad_sequences(sequences, maxlen=200)

    test_sequences = tokenizer.texts_to_sequences(X_test)
    X_test = pad_sequences(test_sequences, maxlen=200)

    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=seed, stratify=y_train)

    y_train = np.array(y_train)
    y_test = np.array(y_test)

    model = create_model()

    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=128,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping]
    )

    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)

    f1.append(f1_score(y_test, y_pred_classes, average='weighted'))

Epoch 1/20
224/224 [==============================] - 49s 206ms/step - loss: 1.0153 - accuracy: 0.4578 - val_loss: 0.9190 - val_accuracy: 0.5498
Epoch 2/20
224/224 [==============================] - 46s 207ms/step - loss: 0.8486 - accuracy: 0.5984 - val_loss: 0.9098 - val_accuracy: 0.5565
Epoch 3/20
224/224 [==============================] - 47s 210ms/step - loss: 0.7090 - accuracy: 0.6724 - val_loss: 0.9389 - val_accuracy: 0.5717
Epoch 4/20
224/224 [==============================] - 46s 206ms/step - loss: 0.6155 - accuracy: 0.7257 - val_loss: 1.0042 - val_accuracy: 0.5697
Epoch 5/20
224/224 [==============================] - 45s 203ms/step - loss: 0.5406 - accuracy: 0.7622 - val_loss: 1.0695 - val_accuracy: 0.5684
Epoch 6/20
224/224 [==============================] - 42s 188ms/step - loss: 0.4904 - accuracy: 0.7818 - val_loss: 1.1944 - val_accuracy: 0.5734
Epoch 7/20
224/224 [==============================] - 40s 179ms/step - loss: 0.4627 - accuracy: 0.7952 - val_loss: 1.2144 - val_ac

In [ ]:
import numpy as np

In [ ]:
from keras.callbacks import EarlyStopping
from sklearn.metrics import f1_score

seeds = [115]
f1=[]

X = df['req-text']
y = pd.factorize(df['age-bracket'])[0]

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(X_train)

    vocab_size = len(tokenizer.word_index) + 1

    sequences = tokenizer.texts_to_sequences(X_train)
    X_train = pad_sequences(sequences, maxlen=200)

    test_sequences = tokenizer.texts_to_sequences(X_test)
    X_test = pad_sequences(test_sequences, maxlen=200)

    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=seed, stratify=y_train)

    y_train = np.array(y_train)
    y_test = np.array(y_test)

    model = create_model()

    early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=128,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping]
    )

    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)

    f1.append(f1_score(y_test, y_pred_classes, average='weighted'))

print(f1)

Epoch 1/20
224/224 [==============================] - 22s 93ms/step - loss: 1.0148 - accuracy: 0.4616 - val_loss: 0.9378 - val_accuracy: 0.5387
Epoch 2/20
224/224 [==============================] - 20s 91ms/step - loss: 0.8383 - accuracy: 0.6092 - val_loss: 0.9192 - val_accuracy: 0.5569
Epoch 3/20
224/224 [==============================] - 20s 90ms/step - loss: 0.6899 - accuracy: 0.6934 - val_loss: 0.9558 - val_accuracy: 0.5632
Epoch 4/20
224/224 [==============================] - 21s 92ms/step - loss: 0.5887 - accuracy: 0.7421 - val_loss: 1.0577 - val_accuracy: 0.5630
Epoch 5/20
224/224 [==============================] - 20s 91ms/step - loss: 0.5236 - accuracy: 0.7761 - val_loss: 1.1238 - val_accuracy: 0.5619
Epoch 6/20
224/224 [==============================] - 21s 92ms/step - loss: 0.4721 - accuracy: 0.7961 - val_loss: 1.2226 - val_accuracy: 0.5519
Epoch 7/20
224/224 [==============================] - 21s 92ms/step - loss: 0.4437 - accuracy: 0.8071 - val_loss: 1.3476 - val_accuracy:

In [ ]:
f1

[0.5589216186232665,
 0.563559178078609,
 0.5721973033738913,
 0.5709944547936916,
 0.5681450579179833,
 0.5537780131059713,
 0.5659133099409086,
 0.5618926935315451]

## Cascata

In [ ]:
dict_class = pd.read_csv('../data/dict-class.csv', encoding='latin-1')

In [ ]:
dict_class

,Unnamed: 0,index,age-bracket,req-text,req-text-clean,Age,glex1,glex2,glex3,c,Age_,c50,c5025,cmean,c75
0,0,12957,a17-30,Necessito das tabelas de Índice de atualização...,necessito tabelas índice atualização contribui...,0,-1.631111,0.517476,0.246113,2,1,2,2,0,0
1,2,873,a17-30,"Prezados, estou elaborando um estudo sobre os ...",prezados elaborando estudo sobre impactos inve...,0,0.784827,4.547806,4.163538,1,1,1,1,1,0
2,11,45511,a31-42,"Prezados, Conforme redirecionamento do Suporte...",prezados conforme redirecionamento suporte sis...,1,-2.090903,-1.919023,-0.459755,3,2,3,3,3,3
3,17,7966,a43-xx,É possível obter uma lista de professores Titu...,possível obter lista professores titulares uni...,2,1.027527,0.866280,0.355635,1,3,1,1,1,0
4,28,44802,a17-30,Gostaria de solicitar duas infomações: 1. Exis...,gostaria solicitar duas infomações 1 existe ca...,0,1.606195,3.874015,1.784264,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2682,9532,24032,a17-30,"Prezados, Gostaria de obter uma cópia digitali...",prezados gostaria obter cópia digitalizada est...,0,1.262396,1.665968,1.044208,1,1,1,1,1,0
2683,9537,34095,a17-30,Gostaria de saber se há vagas para o cargo de ...,gostaria saber vagas cargo nutricionista técni...,0,1.604336,1.761439,0.021796,1,1,1,1,1,1
2684,9538,16968,a17-30,Estou classificado para o IFTO e gostaria de s...,classificado ifto gostaria aproveitado gostari...,0,1.533755,2.029393,1.558002,1,1,1,1,1,1
2685,9540,20928,a17-30,Gostaria de obter informações sobre as obras p...,gostaria obter informações sobre obras paralis...,0,1.798869,3.317710,1.770055,1,1,1,1,1,1


In [ ]:
df['req-text'][873]

'Prezados, estou elaborando um estudo sobre os impactos dos investimentos feitos pelo FGTS nas economias dos municípios brasileiros. Para tanto, necessito de dados detalhados por município acerca dos montantes aplicados em cada área de aplicação (saneamento, habitação e infraestrutura) em cada ano desde 2000. No site do FGTS, há um campo para consulta da contratação diária por município, entretanto, a pesquisa é lenta e fornece os resultados apenas para um município de cada vez. Considerando que temos mais de 5.000 municípios no país, a consulta dessas informações pelo site inviabiliza o estudo pretendido. Além disso, no site não constam informações sobre os valores desembolsados, que são os mais relevantes para a análise, apenas aparecem os dados dos valores contratados. Assim, venho por meio do e-SIC solicitar as seguintes informações, por município e por ano desde 2000 até 2014: valores contratados em cada área de aplicação do FGTS, valores desembolsados em cada área de aplicação, n

In [ ]:
result = df.reindex(dict_class['index'])

In [ ]:
result.shape, dict_class.shape

((2687, 137), (2687, 15))

In [ ]:
def get_age_class(age):
    if age == 'a17-30': return 0
    elif age == 'a31-42': return 1
    else: return 2

In [ ]:
import numpy as np
from sklearn import metrics
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping


In [ ]:
X = df['req-text']
df["Age"] = df['age-bracket'].apply(lambda x: get_age_class(x))
y = df['Age']

df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

y_train = df_train['Age']
X_train = df_train['req-text']

y_test = df_test['Age']
X_test = df_test['req-text']

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index) + 1

sequences = tokenizer.texts_to_sequences(X_train)
X_train = pad_sequences(sequences, maxlen=200)

test_sequences = tokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(test_sequences, maxlen=200)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=115, stratify=y_train)

y_train = np.array(y_train)
y_test = np.array(y_test)

model = create_model()

early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=128,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping]
)

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

df_test['Predict'] = y_pred_classes

print(metrics.f1_score(y_test, y_pred_classes, average='macro'))

Epoch 1/20
224/224 [==============================] - 21s 87ms/step - loss: 1.0180 - accuracy: 0.4623 - val_loss: 0.9362 - val_accuracy: 0.5377
Epoch 2/20
224/224 [==============================] - 20s 88ms/step - loss: 0.8365 - accuracy: 0.6089 - val_loss: 0.9141 - val_accuracy: 0.5513
Epoch 3/20
224/224 [==============================] - 20s 91ms/step - loss: 0.6883 - accuracy: 0.6909 - val_loss: 0.9689 - val_accuracy: 0.5578
Epoch 4/20
224/224 [==============================] - 20s 89ms/step - loss: 0.5868 - accuracy: 0.7404 - val_loss: 1.0235 - val_accuracy: 0.5618
Epoch 5/20
224/224 [==============================] - 20s 90ms/step - loss: 0.5156 - accuracy: 0.7735 - val_loss: 1.1862 - val_accuracy: 0.5656
Epoch 6/20
224/224 [==============================] - 20s 90ms/step - loss: 0.4697 - accuracy: 0.7958 - val_loss: 1.1921 - val_accuracy: 0.5668
Epoch 7/20
224/224 [==============================] - 20s 92ms/step - loss: 0.4402 - accuracy: 0.8073 - val_loss: 1.3047 - val_accuracy:

In [ ]:
dict_class = pd.read_csv('../data/dict-class.csv', encoding='latin-1')

In [ ]:
df_test = df_test.reset_index(drop=True)
dict_class = dict_class.reset_index(drop=True)

df_test['Unnamed: 0'] = df_test['Unnamed: 0'].astype(int)
dict_class['Unnamed: 0'] = dict_class['Unnamed: 0'].astype(int)

compare_test = df_test[df_test['Unnamed: 0'].isin(dict_class['Unnamed: 0'])]

In [ ]:
compare_test = compare_test
compare_dict = dict_class[['Unnamed: 0', 'c5025']]

compare = compare_test.merge(compare_dict, left_on='Unnamed: 0', right_on='Unnamed: 0')

In [ ]:
compare.head()

,Unnamed: 0,index,age-bracket,req-text,req-text-clean,Age,glex1,glex2,glex3,c,c50,c5025_x,cmean,c75,Predict,c5025_y
0,6,5476,a17-30,"Olá, Estou realizando meu trabalho de conclusã...",olá realizando trabalho conclusão curso estuda...,0,2.593333,3.206568,1.563178,0,0,0,0,0,0,0
1,10,3721,a43-xx,Gostaria de solicitar dados do consumo de ener...,gostaria solicitar dados consumo energia elétr...,2,3.444323,2.554219,0.308080,0,0,0,0,0,2,0
2,11,23181,a31-42,"Boa tarde, gostaria da quantidade de farmácias...",boa tarde gostaria quantidade farmácias popula...,1,1.050923,2.238745,1.804996,0,0,0,0,-1,0,0
3,13,971,a43-xx,"Prezado Sr. Geól. Paulo Ribeiro, segue comprov...",prezado sr geól paulo ribeiro segue comprovant...,2,-1.326335,-2.815001,-3.232350,2,2,2,2,2,1,2
4,16,5098,a17-30,"Prezados, Gostaria de solicitar o seguinte: a)...",prezados gostaria solicitar seguinte lista cla...,0,4.821409,5.661898,2.403441,0,0,0,0,0,0,0


In [ ]:
dict_class.shape, compare.shape, compare_test.shape, df_test.shape

((2752, 15), (545, 8), (545, 6), (9553, 6))

In [ ]:
metrics.f1_score(compare['Age'], compare[f'Predict'], average='macro'), metrics.f1_score(compare['Age'], compare['c5025_x'], average='macro')

(0.6335877862595419, 0.6281352235550709)

In [ ]:
compare.head(), compare.shape

(   Unnamed: 0 age-bracket                                           req-text  \
 0        3721      a43-xx  Gostaria de solicitar dados do consumo de ener...   
 1        4461      a31-42  Bom dia, Eu tenho 35 anos e trabalhei por muit...   
 2        4143      a31-42  Solicito, por gentileza, o Quadro de Referênci...   
 3         750      a17-30    Solicito cópia do processo 48500.002286/2004-14   
 4        2357      a17-30  Os dados solicitados são descritos em anexo, s...   
 
                                       req-text-clean  Age  Predict  c5025  \
 0  gostaria solicitar dados consumo energia elétr...    2        2      1   
 1  bom dia 35 anos trabalhei tempo informalidade ...    1        0      1   
 2  solicito gentileza quadro referência servidore...    1        1      3   
 3          solicito cópia processo 48500002286200414    0        2      1   
 4  dados solicitados descritos anexo solicitase d...    0        0      2   
 
    Age_  
 0     2  
 1     3  
 2     3 

In [ ]:
merged_df = df_test.merge(compare[['Unnamed: 0', 'c5025_x']], on='Unnamed: 0', how='left')


In [ ]:
merged_df['Predict'] = merged_df['c5025_x'].combine_first(merged_df['Predict'])

In [ ]:
metrics.f1_score(merged_df['Age'], merged_df[f'Predict'], average='macro'), metrics.f1_score(df_test['Age'], df_test['Predict'], average='macro')

(0.570292054851879, 0.5718622422275725)